<a href="https://colab.research.google.com/github/Sammy-Shieunda/Dysgraphia-screening-capstone/blob/main/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading and Quality Checks
### Early Dysgraphia Screening from Handwriting — Potential Dysgraphia Handwriting Dataset (PDM)

**CRISP-DM phase:** Data Understanding

**Purpose of this notebook**

Before any modelling can happen, we need to be confident about *what* is actually in the dataset. For a tabular dataset you'd reach for `df.shape` and `df.describe()` to get an instant feel for size, structure and distribution. Images don't have rows and columns of numbers in the same way, so this notebook builds the **image equivalent**:

| Tabular data (`pandas`) | Image data (this notebook) |
|---|---|
| `df.shape` | total image count, count per class |
| `df.dtypes` | file format, colour mode |
| `df.describe()` | distribution of width, height, aspect ratio, file size |
| `df.isna().sum()` | corrupted / unreadable files |
| `df.duplicated().sum()` | duplicate images (exact byte-for-byte) |

Specifically, this notebook:

1. **Loads the dataset** into a single tabular index (one row per image) so every later step can just operate on a DataFrame.
2. **Counts images per class** (`Potential Dysgraphia` vs `Low Potential Dysgraphia`) — this tells us whether the classes are balanced, which directly affects how we should evaluate the model later (recall on the minority/at-risk class is the metric this project cares about most).
3. **Checks colour mode and dimensions** — every image needs to be the same shape and channel depth before it can be fed to a CNN, so we need to know how much resizing/normalising work lies ahead.
4. **Checks for corrupted files** — a single unreadable image can crash a training loop hours into a run if it isn't caught here first.
5. **Checks for duplicate files** — duplicates that end up split across the train/validation/test sets cause **data leakage**, which quietly inflates evaluation metrics.
6. **Produces a Data Quality Summary** — a single consolidated report you can drop straight into your project documentation and reference when your lecturer asks "how do you know the data is clean?"

> **Before running:** unzip `DATASET_DYSGRAPHIA_HANDWRITING.zip` and point `DATASET_ROOT` below at the folder that directly contains the two class subfolders (`Low Potential Dysgraphia/` and `Potential Dysgraphia/`).


## 1. Setup and Imports

We only need standard libraries plus `Pillow` (image handling) and `pandas`/`matplotlib` (the same tools you'd use for tabular EDA). Keeping the dependency list small makes this easy to run in Colab or a plain local environment.


In [1]:
import os
import hashlib
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Make plots a bit more readable by default
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_colwidth", 60)


In [9]:
# CONFIG — update this path to match where you unzipped the dataset

DATASET_ROOT = Path("DATASET DYSGRAPHIA HANDWRITING")  # folder containing the two class subfolders

# Sanity check that the path exists before we go any further
assert DATASET_ROOT.exists(), (
    f"Could not find '{DATASET_ROOT}'. Update DATASET_ROOT to point at the "
    f"unzipped dataset folder (the one that directly contains the class subfolders)."
)

CLASS_DIRS = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(CLASS_DIRS)} class folders:")
for d in CLASS_DIRS:
    print(" -", d.name)


AssertionError: Could not find 'DATASET DYSGRAPHIA HANDWRITING'. Update DATASET_ROOT to point at the unzipped dataset folder (the one that directly contains the class subfolders).